In [1]:
# Imports
import torch
import torch.nn as nn
import pandas as pd

In [3]:
sentiments = {
    "LABEL_0": "Bearish",
    "LABEL_1": "Bullish", 
    "LABEL_2": "Neutral"
}

In [4]:
# Load Train and Valid

train = pd.read_csv("sent_train.csv")
valid = pd.read_csv("sent_valid.csv")

In [6]:
# Check the are loaded correctly

display(train.head())
display(valid.head())

,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0


,text,label
0,$ALLY - Ally Financial pulls outlook https://t...,0
1,"$DELL $HPE - Dell, HPE targets trimmed on comp...",0
2,$PRTY - Moody's turns negative on Party City h...,0
3,$SAN: Deutsche Bank cuts to Hold,0
4,$SITC: Compass Point cuts to Sell,0


In [ ]:
# X and y data separation

X_train = train["text"]
y_train = train["label"]

X_valid = valid["text"]
y_valid = valid["label"]

In [ ]:
# LSTM Architecture
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        embedded = self.embedding(x)  
        output, (h_n, c_n) = self.lstm(embedded)
        
        pooled = output.mean(dim=1) # Mean pooling
        logits = self.fc(pooled)
        return logits

In [ ]:
# Get every word
words = set()

# Get all types
for text in X_train:
    for word in text.lower().split():
        words.add(word)

# Map words to index
vocabulary = {w: idx for idx, w in enumerate(words)}

# Convert words to index
def words_to_idx(text):
    return [vocabulary.get(word) for word in text.lower().split()]

In [ ]:
# Get vocabulary size
vocab_size = len(vocabulary)

# Get number of classes
num_classes = y_train["label"].nunique()

# Label to sentiment
def label_to_sentiment(label):
    if label == 0:
        return "Bearish"
    elif label == 1:
        return "Bullish"
    elif label == 2:
        return "Neutral"

# Embed dimensions    
embed_dim = 100